In [ ]:
import json, time, tqdm, os, pickle, regex as rx, pandas as pd, random

from collections import Counter
from openai import OpenAI
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from islab.nlp.cancer_registry.datasets.utils import CRCPlainTextReader
from dateutil.parser import parse
from pathlib import Path
from sklearn.metrics import classification_report
from llama_index.core.prompts.base import PromptTemplate
from llama_index.core import VectorStoreIndex, Settings, StorageContext, load_index_from_storage
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler
from llama_index.core.schema import NodeWithScore
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.tools.tool_spec.base import BaseToolSpec
from islab.nlp.cancer_registry.agents.llamaindex import CancerRegistryReActAgent
from llama_index.core.llms import ChatMessage

In [ ]:
HospitalID = "KMUH"
# HospitalID = "CMUH"
# HospitalID = "KAFGH_C"
# HospitalID = "PIH"

# retrieverModel = "BM25"
# retrieverModel = "GIST-large-Embedding-v0"
# retrieverModel = "multilingual-e5-large-instruct"
# retrieverModel = "b1ade-embed"
# retrieverModel = "bge-fr-en"
# retrieverModel = "Solon-embeddings-large-0.1"
# retrieverModel = "GIST-small-Embedding-v0"
# retrieverModel = "GIST-Embedding-v0"
# retrieverModel = "mxbai-embed-large-v1"
# retrieverModel = "UAE-Large-V1"
# retrieverModel = "e5-base-4k"
retrieverModel = "bge-large-en-v1.5"

llmModel = "Llama-3.1-8B-Instruct"

rerankModel = None

queryMode = "RAG"

chunkSize = 512

topkMerage = False
# topkMerage = True

top_k = 3

filePatientPath = f'datasets/{HospitalID}/testset'
fileAnsPath = f'datasets/{HospitalID}/Report_Answer.tsv'

In [ ]:
if retrieverModel in {"BM25"}:
    Settings.embed_model = None
else:
    Settings.embed_model = HuggingFaceEmbedding(model_name=f"./.models/Retriever/{retrieverModel}")
Settings.llm = OpenAILike(
    model=llmModel,
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
    temperature=0.01,
    is_chat_model=False,
    is_function_calling_model=False,
    strict=False,
    max_tokens=256
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"./.models/LLM/Mistral-7B-v0.3")

In [ ]:
def prepare_time_line_data(docs):
	tl_info = {}
	tl_info['events']=[]
	for doc in docs:
		report = {}
		date = parse(doc.metadata['date'])
		report["start_date"] = {"year": date.year, "month": date.month, "day": date.day}
		report["text"] = {
			"text": doc.text.replace('\n', '<br/>')
		}
		tl_info['events'].append(report)	
	return tl_info

def load_dataset(path):
	reader = CRCPlainTextReader()
	patients = reader.load_data(path, HospitalID)
	patient_list = {}
	for pid in patients.keys():	
		patient_list[Path(pid).stem]=[prepare_time_line_data(patients[pid]), patients[pid]]	
	return patient_list

def load_gold_answers(path):
	df = pd.read_csv(path, sep='\t')
	return df

patient_list = load_dataset(filePatientPath)
gold_ans = load_gold_answers(fileAnsPath)

In [ ]:
ignore_list= ["f02af613e39ea56439b0f07fb8ea0df6", "f04b8e681dd034ffb824ee02bd93430b", "f04b8e681dd034ffb824ee02bd93430b",
              "f071e89186538c0585b39de479bb3375", "f07c69ef6cddb16e3547dbffe378be7e", "0f038609b08ea9930f097d67f210e90",
            "f11e6add274bb0668690d5cfff234e9c","f177a11156aaa36944e25fcaf40093d1","f192b5f188924d2cccce6b937e7574ad",
            "f1d68b0e1f2301333e3c45651908c149","f21175384d528a51d59a1ffbeba4d418","f2656a1ca6fb27674b071f086e4bf0df",
            "f2bb03d7f2f892c7f1f734c61950ab66","f309cc0d52849d12a80573ebacdb1710","f314c3a9c3e88761253b092c2157576d",
            "f355449a1620d556250596f416bfc8bd","f37044ce5da8c8148bc85633b9c56d5f", "f3816c6c15c84e837a670848b988fdd4",
            "f3aa8245b1d6fbfd922231317da5f781","f4126d436e7bf23786cc0c2da0b29912","f48c0f7cd40416065a57bd653aed5386",
            "f498820fa6f4d5aab7bec0ffc54c8718","f4a4effc1ad5abb8db7b3dae1fb0bb85","f4cca1760d5730191772b5f4445f88d7",
            "f51013595fdd625b59dcc20afe1ab761","f57d767e6dec9e86601ba6da514c4f4d","f5b102b89e69163aafb619c0661d046b",
            "f5db3c7be6a4e351c8416f08f6fbe7aa","f5e4ce1a4a0de092981846809ac394b0","f67a3b7b00a11b267eb909f672ead14b",
            "f853f6f49c9d40b161bbe21bfb401f7a","f8b85f1445055c343ad02370f8b7a06a","f8be7b390f75f4f43fd214365a94fb4e",
            "f8edffcb3f9d19e7034ead9d544a215c","f971d04e879cc0d4bbb9ee0a2453b5cf","f9d3cec69f0725005087b71e5f45f5fe",
            "fa3c3929227f3eeb748f38629f425dad","fa9ac9dcc3d37f26273aacb963c023c5","faf1204786135affc70142fb9932f4c3",
            "faf4c5b21684af29046808829935ccd7","fb25af4d3985924f61edf7c2b7261891","fd188be319a4c3fef06cc474a5807144",
            "fd7cd9931a7e7bc4729ef1780613bcd9","ff937072bc143d00708db2cd48e4a7e6","ffccb1d17ea1e16595bcd36e354aa520",
            "f02087e255e12e62bbcb3b7994f2939c"]

def select_registry_item(gold_ans, item_name, ignore_list = []):
    patient_list = load_dataset(filePatientPath)
    gold_ans = load_gold_answers(fileAnsPath)    
    ans = gold_ans[['CHART_NO', item_name]]
    ans[item_name] = ans[item_name].astype(str)
    patient_list = {k: v for k, v in patient_list.items() if k not in ignore_list}
    
    for k in patient_list.keys():
        if HospitalID in {"KMUH"}:
            indexs = ans[ans['CHART_NO'] == k].index
        elif HospitalID in {"PIH", "KAFGH_C", "KAFGH_L", "CMUH"}:
            indexs = ans[ans['CHART_NO'] == int(k.strip())].index
        value = ans.iloc[indexs][item_name]
        patient_list[k].append({item_name: value.values.tolist()})
    ans = ans[~ans['CHART_NO'].isin(ignore_list)]
    return ans, patient_list

def select_registry_items(gold_ans, item_names, ignore_list = []):
    patient_list = load_dataset(filePatientPath)
    gold_ans = load_gold_answers(fileAnsPath)    
    ans = gold_ans[['CHART_NO'] + item_names]
    ans[item_names] = ans[item_names].astype(str).apply(lambda x: x.str.strip())
    patient_list = {k: v for k, v in patient_list.items() if k not in ignore_list}
    
    for k in patient_list.keys():
        if HospitalID in {"KMUH"}:
            indexs = ans[ans['CHART_NO'] == k].index
        elif HospitalID in {"PIH", "KAFGH_C", "KAFGH_L", "CMUH"}:
            indexs = ans[ans['CHART_NO'] == int(k.strip())].index
        values = ans.iloc[indexs][item_names]
        patient_list[k].append({item_name: values[item_name].values.tolist() for item_name in item_names})  
    ans = ans[~ans['CHART_NO'].isin(ignore_list)]
    return ans, patient_list

In [ ]:
def create_node_parser(callback_manager = None):
    if chunkSize == 1024:
        chunk_sizes = [1024, 512, 256]
    elif chunkSize == 512:
        chunk_sizes = [512, 256, 128]
    elif chunkSize == 256:
        chunk_sizes = [256, 128]
    return HierarchicalNodeParser.from_defaults(chunk_sizes=chunk_sizes, callback_manager= callback_manager)

node_parser = create_node_parser()

In [ ]:
ccr_histology_template ='''Your primary task is to accurately curate the histology code for patients diagnosed with colorectal cancer. This involves meticulously reviewing and analyzing the given medical record of the patient to determine the most appropriate histology code. Your decision should be based on a thorough assessment of clinical findings and any relevant medical descriptions that shed light on the histological nature of the colorectal cancer. It is essential to use only the information provided in the document at hand.
When responding, you are required to select the most appropriate option from the following standard codes, which represents different histology types of colorectal cancer:

- 8000: Use this code if the histology type is "Neoplasm, malignant", "Blastoma, NOS", "Tumor, malignant, NOS", "Cancer", "Malignancy", "Unclassified tumor, malignant".
- 8010: Use this code if the histology type is "Carcinoma in situ, NOS", "Intraepithelial carcinoma, NOS", "Carcinoma, NOS", "Epithelial tumor, malignant", or "Heterotopia-associated carcinoma".
- 8013: Use this code if the histology type is "Large cell neuroendocrine carcinoma", or "Combined large cell neuroendocrine carcinoma".
- 8014: Use this code if the histology type is "Large cell carcinoma with rhabdoid phenotype".
- 8020: Use this code if the histology type is "Carcinoma, undifferentiated, NOS", "Carcinoma, poorly differentiated, NOS", "Anaplastic undifferentiated carcinoma", or "Dedifferentiated carcinoma".
- 8070: Use this code if the histology type is "squamous cell carcinoma in situ, NOS", "epidermoid carcinoma in situ, NOS", "intraepidermal carcinoma, NOS", "intraepithelial squamous cell carcinoma", "squamous cell carcinoma, NOS", "squamous cell carcinoma, usual type", "epidermoid carcinoma, NOS", "squamous carcinoma", or "squamous cell epithelioma".
- 8120: Use this code if the histology type is "Transitional cell carcinoma in situ", "Urothelial carcinoma in situ", "Transitional cell carcinoma, NOS", "Urothelial carcinoma, NOS", or "Transitional carcinoma".
- 8140: Use this code if the histology type is "Adenocarcinoma in situ, NOS", "Adenocarcinoma, NOS", "Adenocarcinoma, usual type", "Carcinoma of Skene, Cowper and Littre Glands", "Endolymphatic sac tumor", "Parathyroid carcinoma", or "Acinar adenocarcinoma of prostate".
- 8210: Use this code if the histology type is "Adenocarcinoma in situ in adenomatous polyp", "Adenocarcinoma in situ in a polyp, NOS", "Carcinoma in situ in a polyp, NOS", "Adenocarcinoma in situ in polypoid adenoma", "Adenocarcinoma in situ in tubular adenoma", "Carcinoma in situ in adenomatous polyp", "Adenocarcinoma in adenomatous polyp", "Adenocarcinoma in a polyp, NOS", "Carcinoma in a polyp, NOS", or "Adenocarcinoma in polypoid adenoma".
- 8211: Use this code if the histology type is "Tubular adenocarcinoma", or "Tubular carcinoma".
- 8244: Use this code if the histology type is "Mixed adenoneuroendocrine carcinoma", "Combined carcinoid and adenocarcinoma", "Combined/mixed carcinoid and adenocarcinoma", "Composite carcinoid", "MANEC", or "Mixed carcinoid-adenocarcinoma"
- 8246: Use this code if the histology type is "Neuroendocrine carcinoma, NOS", or "Poorly differentiated neuroendocrine neoplasm".
- 8261: Use this code if the histology type is "Adenocarcinoma in situ in villous adenoma", or "Adenocarcinoma in villous adenoma".
- 8263: Use this code if the histology type is "Adenocarcinoma in situ in tubulovillous adenoma", "Adenocarcinoma in tubulovillous adenoma", "Villoglandular carcinoma", "Villoglandular variant of endometrioid adenocarcinoma", "Papillotubular adenocarcinoma", or "Tubulopapillary adenocarcinoma".
- 8480: Use this code if the histology type is "Mucinous adenocarcinoma", "Acinar adenocarcinoma, mucinous variant", "Colloid adenocarcinoma", "Colloid carcinoma", "Gelatinous adenocarcinoma", "Gelatinous carcinoma", "Mucinous carcinoma", "Mucoid adenocarcinoma", "Mucoid carcinoma", "Mucous adenocarcinoma", "Mucous carcinoma", "Pseudomyxoma peritonei with unknown primary site", "Mucinous tubular and spindle cell carcinoma".
- 8490: Use this code if the histology type is "Signet ring cell carcinoma", "Signet ring cell adenocarcinoma", "Acinar adenocarcinoma, signet ring-like variant", "Mucinous carcinoma, signet ring cell type", "Poorly cohesive carcinoma", or "Signet ring cell/histiocytoid carcinoma".
- 8510: Use this code if the histology type is "Medullary carcinoma, NOS", "Medullary adenocarcinoma", or "Medullary-like carcinoma".

Your response should be limited to one of these codes, aligning with the most accurate and specific histological classification of the cancer case. For example, if the case is pathologically diagnosed as 'Adenocarcinoma in situ in adenomatous polyp', your response should be in the following JSON format:

{
"explain": "[Your explanation here based on the document]",
"cite": "[Cited passages from the document used for your decision]",
"histology": "8210"
}

Alternatively, if the case is diagnosed as 'Adenocarcinoma in situ' without further specification, your response should be:

{
"explain": "[Your explanation here based on the document]",
"cite": "[Cited passages from the document used for your decision]",
"histology": "8140"
}
'''

ccr_node_examine_template = '''Your task is to record the total number of regional lymph nodes examined by the pathologist in each case. This data is crucial for understanding the extent of lymph node examination in cancer cases. When coding, focus on the total count of lymph nodes examined, regardless of whether they are positive or negative for malignancy.
Here are three examples to guide you:
Example 1: The report states 'Lymph node, regional, lymphadenectomy, negative for malignancy (0/20)'. The code for this case is '20', representing the total number of lymph nodes examined is 20.
Example 2: The report states 'Lymph node, apical, lymphadenectomy, negative for malignancy (0/9)'. In this case, the correct code is '9', indicating the total count of lymph nodes examined is 9.
Example 3: The report states 'Lymph node regional lymphadenectomy negative for malignancy (0/16). Lymph node apical lymphadenectomy negative for malignancy (0/2).' In this case, the correct code is '18', which is determined by 16 + 2 = 18 indicating the total count of lymph nodes examined is 18 including 16 regional lymph nodes and 2 apical lymph nodes.

Your response should be in the following JSON format and limited to the number of nodes described in the report.

{
"cite": "Cited passages from the document used for your decision",
"explain": "Your explanation here based on the document",
"examined_nodes": "Number of examined nodes"
}
'''

ccr_node_positive_template = '''Your task is to record the total number of positive regional lymph nodes reported by the pathologist in each case. This data is crucial for understanding the extent of positive lymph nodes in cancer cases. When coding, focus on the total count of positive lymph nodes, and ignore nodes that are negative for malignancy.
Here are two examples to guide you:
Example 1: The report states 'Lymph node, regional, lymphadenectomy, negative for malignancy (0/20)'. The code for this case is '0', representing the total number of positive lymph nodes is 0.
Example 2: The report states 'Lymph node regional lymphadenectomy negative for malignancy (0/16). Lymph node apical lymphadenectomy negative for malignancy (0/2).' In this case, the correct code is '0', which is determined by 0 + 0 = 0 indicating the total count of lymph nodes examined is 0 including 0 regional lymph nodes and 0 apical lymph nodes.

Your response should be in the following JSON format and limited to the number of nodes described in the report.

{
"cite": "Cited passages from the document used for your decision",
"explain": "Your explanation here based on the document",
"positive_nodes": "Number of positive nodes"
}
'''

ccr_path_tnm_template = '''Your task is to response the cancer staging information in terms of PATH_T, PATH_N, and PATH_M based on the pathological TNM criteria. The TNM staging system is crucial for assessing trends in cancer treatment and control. It is used by physicians for prognosis estimation, treatment planning, evaluation of therapies, outcome analysis, follow-up planning, and assessing early detection results. Pathological staging is primarily determined by pathologists, with input from other medical specialists. It is usually written in the format of T(PATH_T code)N(PATH_N code)[M(PATH_M code)], where the PATH_M code may be omitted. For example, in the description 'T1b1N2cM0', the PATH_T, PATH_N, and PATH_M codes are 1b, 2c, and 0, respectively. In cases where no explicit description is provided by physicians, you should make informed judgments based on available information in the medical records.

Here is the detail definition for the three codes.
- PATH_T: This code indicates the size or extent of the primary tumor.
-- Available codes: X, 0, A, IS, ISU, ISD, ISDC, ISPA, ISLA, 1M, 2, 2A, 2A1, 2A2, 2B, 2C, 2D, 3, 3A, 3B, 1, 1A, 1A1, 1A2, 1B, 1B1, 1B2, 1B3, 1C, 1C1, 1C2, 1C3, 1D, 3C, 3D, 4, 4A, 4B, 4C, 4D, 4E, 8888 (Not applicable), 9999 (Not recorded)
- PATH_N: This code represents the presence and extent of regional lymph node metastasis.
-- Available codes: X, 0, 0A, 0B, 0C, 0D, 1, 1A, 1AS, 1B, 1C, 1M, 2M, 2, 2A, 2B, 2C, 3, 3A, 3B, 3C, 888 (Not applicable), 999 (Not recorded)
- PATH_M: This code indicates the presence of distant metastasis.
-- Available codes: 1, 1A, 1A0, 1A1, 1B, 1B0, 1B1, 1C, 1C0, 1C1, 1D, 1D0, C, CA, CA0, CA1, CB, CB0, CB1, CC, CC0, CC1, CD, CD0, CD1, CE, 888 (Not applicable), 999 (Not recorded), B (if there is no mention of distant metastasis, the M stage is assumed to be 'B').

Your response should be in the following JSON format and limited to the number of nodes described in the report.

{
"cite": "Cited passages from the document used for your decision",
"explain": "Your explanation here based on the document",
"path_t": "PATH_T code",
"path_n": "PATH_N code",
"path_m": "PATH_M code"
}

Please ensure your response is limited to one of these specific codes. For example, if the report states 'pT3 N1c', your coding results in this case should be:
{
"cite": "pT3 N1c",
"explain": "The report states 'T3N1c'. The corresponding PATH_T/N code is 3/1C. The PATH_M code is B indicating that the PATH_M code cannot curate",
"path_t": "3",
"path_n": "1C",
"path_m": "B"
}
'''

ccr_grade_template = '''Your task is to extracts the histologic grade (Grade) for cancer registry coding.
When responding, you are required to select the most appropriate option from the following standard codes, which represents different grade types of colorectal cancer:
- 1: Use this code if the grade type is 'well differentiated, low grade'.
- 2: Use this code if the grade type is 'moderately differentiated'.
- 3: Use this code if the grade type is 'poorly differentiated, high grade'.
- 4: Use this code if the grade type is 'undifferentiated, anaplastic'.
- If the text explicitly contains "Grade 1/2/3/4" or "Grade I/II/III/IV", directly use that number. 

Your response should be limited to one of these codes, aligning with the most accurate and specific histological classification of the cancer case. For example, if the case is pathologically diagnosed as 'moderately differentiated', your response should be in the following JSON format:

{
"explain": "[Your explanation here based on the document]",
"cite": "[Cited passages from the document used for your decision]",
"grade": "2"
}

Alternatively, if the case is diagnosed as 'Grade III' without further specification, your response should be:

{
"explain": "[Your explanation here based on the document]",
"cite": "[Cited passages from the document used for your decision]",
"grade": "3"
}
'''

if HospitalID in {"PIH"}:
    ccr_tumor_size_template = '''Your task is to extract the maximum tumor size mentioned in the pathological report for each colorectal cancer case. Tumor size is a critical factor in staging and treatment decisions, and should be recorded in centimeters (mm) whenever possible.
    
    Here are examples for clarification:
    If the case description says as 'Tumor measured 2.5 x 2.0 x 1.5 cm.'. The maximum tumor size is '2.5' cm.Therefore, you must convert the value from centimeters (cm) to millimeters (mm). The final answer should be 25 mm. Your response should be in the following JSON format:
    {
    "cite": "Cited passages from the document used for your decision",
    "explain": "Your explanation here based on the document",
    "tumor_size": "25"
    }
    
    Alternatively, if the description says as 'The mass is approximately 18 mm in the largest dimension.'. The tumor size is '18' millimeter (mm). Your response should be in the following JSON format:
    {
    "cite": "Cited passages from the document used for your decision",
    "explain": "Your explanation here based on the document",
    "tumor_size": "18"
    }
    '''
else:
    ccr_tumor_size_template = '''Your task is to extract the maximum tumor size mentioned in the pathological report for each colorectal cancer case. Tumor size is a critical factor in staging and treatment decisions, and should be recorded in centimeters (cm) whenever possible.
    
    Here are examples for clarification:
    If the case description says as 'Tumor measured 2.5 x 2.0 x 1.5 cm.'. The maximum tumor size is '2.5' cm. Your response should be in the following JSON format:
    {
    "cite": "Cited passages from the document used for your decision",
    "explain": "Your explanation here based on the document",
    "tumor_size": "2.5"
    }
    
    Alternatively, if the description says as 'The mass is approximately 18 mm in the largest dimension. The tumor size is '18' mm. Your response should be in the following JSON format:
    {
    "cite": "Cited passages from the document used for your decision",
    "explain": "Your explanation here based on the document",
    "tumor_size": "1.8"
    }
    '''

In [ ]:
histology_template = """Your task is to provide the 'histology coding' for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
- In cases where the report is not a pathology report or lacks sufficient information to ascertain the histology type, your response should be 'NA'. This indicates that histology coding cannot be determined from the provided document.
- If the report is a pathology report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate histology code.
- If multiple histology types are mentioned, choose only the most appropriate single code that best represents the diagnosis.
"""

examined_nodes_template = """Your task is to provide the 'examined node coding' for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
- If the report is a report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate code.
- In cases where the report lacks sufficient information to ascertain the number of examined nodes, your response should be '99'. This indicates that examined node coding cannot be determined from the provided document.
- In cases where regional lymph nodes were surgically removed, but the number of lymph nodes is unknown/not stated and not documented as a sampling or dissection, your response should be '98'. This indicates that nodes were examined, but the number is unknown.
"""

positive_nodes_template = """Your task is to provide the 'positive node coding' for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
- If the report is a report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate code.
- In cases where the report lacks sufficient information to ascertain the number of positive  nodes, your response should be '99'. This indicates that examined node coding cannot be determined from the provided document.
- In cases where regional lymph nodes were surgically removed, but the number of lymph nodes is unknown/not stated and not documented as a sampling or dissection, your response should be '98'. This indicates that nodes were examined, but the number is unknown.
"""

path_tnm_template = """Your task is to provide the 'pathology TNM coding' for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
- If the report is a report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate code.
- In cases where the report lacks sufficient information to ascertain the patology TNM coding, your response should be 'NA' indicating that the pathology TNM coding cannot be determined from the provided document.
"""

grade_template = """Your task is to provide the 'grade coding' for patients diagnosed with colorectal cancerr. Your response should be based on the nature of the report:
- In cases where the report lacks sufficient information to the grade coding, your response should be 'NA' indicating that the grade coding cannot be detemined from the provided document.
- If multiple mentions exist, use the highest Grade.
"""

if HospitalID in {"PIH"}:
    tumor_size_template = """Your task is to provide the 'tumor size coding' for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
    - If the report is a report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate code.
    - If multiple dimensions are provided, record the largest single measurement.
    - If the tumor is described as unmeasurable or not found, return "999".
    """
else:
    tumor_size_template = """Your task is to provide the 'tumor size coding' for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
    - If the report is a report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate code.
    - If multiple dimensions are provided, record the largest single measurement.
    - If sizes are given in millimeters (mm), convert them to centimeters (cm) by dividing by 10.
    - If the tumor is described as unmeasurable or not found, return "99.9".
    """

In [ ]:
if llmModel in {"Mistral-7B-v0.3", "Mistral-7B-Instruct-v0.3", "Llama-3.1-8B", "Qwen3-8B-Base", "Qwen3-8B", "DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-7B"}:
    qa_prompt_tmpl_str = """Context information is below.
    ---------------------
    {context_str}
    ---------------------
    Given the context information and the following knowledge and command, answer the query.
    Knowledge and command: {cancer_registry_kg}
    Query: {query_str}
    Answer:\
    """
elif llmModel in {"Llama-3.1-8B-Instruct"}:
    qa_prompt_tmpl_str = """Context information is below.
    ---------------------
    {context_str}
    ---------------------
    Given the context information and the following knowledge and command, answer the query.
    Knowledge and command: {cancer_registry_kg}
    Answer:
    """
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

refine_prompt_tmpl_str = """The original query is as follows: {query_str}
We have provided an existing answer: {existing_answer}
Knowledge and command: {cancer_registry_kg}
We have the opportunity to refine the existing answer (only if needed) with some more context below.
---------------------
{context_msg}
---------------------
Given the new context, refine the original answer to better answer the query. If the context isn't useful, return the original answer.
Refined Answer:
"""
refine_prompt_tmpl = PromptTemplate(refine_prompt_tmpl_str)

In [ ]:
class CRCRetrievalEngineToolSpec(BaseToolSpec):
    def __init__(
        self,
        doc,
        idx,
        nodes,
        similarity_top_k = top_k,
        rerank = None
    ) -> None:
        """Initialize with parameters."""
        file_path = Path(doc.metadata["file_path"])
        file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
        vi_out_path = f".data/{HospitalID}/{chunkSize}/{retrieverModel}/indexes/{file_id}_{idx}"

        self.nodes = nodes

        if not os.path.exists(vi_out_path):
            leaf_nodes = get_leaf_nodes(nodes)        
            storage_context = StorageContext.from_defaults()
            storage_context.docstore.add_documents(nodes)
            self._index = VectorStoreIndex(
                    leaf_nodes,
                    storage_context=storage_context
                )
            self._index.storage_context.persist(persist_dir=vi_out_path)
        else:
            self._index = load_index_from_storage(
                StorageContext.from_defaults(persist_dir=vi_out_path),
            )
        self.similarity_top_k = similarity_top_k
        if rerankModel != None:
            self.rerank = SentenceTransformerRerank(top_n=3, model=f"./.models/Rerank/{rerankModel}")
    
    def get_histology_fn(self, query: str) -> str:
        """Extract the histology information.

        Useful for answering questions related to the histology of colorectal cancer patients
        
        Args:
            query (str): The query for the extraction purpose
        """

        if retrieverModel == "BM25":
            startTime = time.time()
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
            endTime = time.time()
        else:
            startTime = time.time()
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            retrieved_nodes = retriever.retrieve(query)
            endTime = time.time()
        
        return endTime - startTime
    
    def get_grade_fn(self, query: str) -> str:
        if retrieverModel == "BM25":
            startTime = time.time()
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
            endTime = time.time()
        else:
            startTime = time.time()
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            retrieved_nodes = retriever.retrieve(query)
            endTime = time.time()
        
        return endTime - startTime
    
    def get_examined_nodes_fn(self, query: str) -> str:
        if retrieverModel == "BM25":
            startTime = time.time()
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
            endTime = time.time()
        else:
            startTime = time.time()
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            retrieved_nodes = retriever.retrieve(query)
            endTime = time.time()
        
        return endTime - startTime
    
    def get_positive_nodes_fn(self, query: str) -> str:
        if retrieverModel == "BM25":
            startTime = time.time()
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
            endTime = time.time()
        else:
            startTime = time.time()
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            retrieved_nodes = retriever.retrieve(query)
            endTime = time.time()
        
        return endTime - startTime
    
    def get_path_tnm_fn(self, query: str) -> str:
        if retrieverModel == "BM25":
            startTime = time.time()
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
            endTime = time.time()
        else:
            startTime = time.time()
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            retrieved_nodes = retriever.retrieve(query)
            endTime = time.time()
        
        return endTime - startTime
    
    def get_tumor_size_fn(self, query: str) -> str:
        if retrieverModel == "BM25":
            startTime = time.time()
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
            endTime = time.time()
        else:
            startTime = time.time()
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            retrieved_nodes = retriever.retrieve(query)
            endTime = time.time()
        
        return endTime - startTime

In [ ]:
def build_analyze_per_doc(doc, idx, type, nodes, answer, target_outcome_list = None):
    crc_tool_spec = CRCRetrievalEngineToolSpec(doc, idx, nodes)
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)
    if rerankModel == None:
        if top_k == 1:
            analysis_file_path = f".retrieverTime/{HospitalID}/Top-1/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}_{idx}"
        elif top_k == 3:
            analysis_file_path = f".retrieverTime/{HospitalID}/RAG/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}_{idx}"
        elif top_k == 6:
            analysis_file_path = f".retrieverTime/{HospitalID}/Top-6/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}_{idx}"
    elif rerankModel != None:
        analysis_file_path = f".retrieverTime/{HospitalID}/RAG(Rerank)/{rerankModel}/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}_{idx}"
    
    if os.path.exists(analysis_file_path):
        return
        
    Path(analysis_file_path).parent.mkdir(parents=True, exist_ok=True)

    if type == 'histology':
        queryTime = crc_tool_spec.get_histology_fn(histology_template)
    elif type =='grade':
        queryTime = crc_tool_spec.get_grade_fn(grade_template)
    elif type =='examined_nodes':
        queryTime = crc_tool_spec.get_examined_nodes_fn(examined_nodes_template)
    elif type =='positive_nodes':
        queryTime = crc_tool_spec.get_positive_nodes_fn(positive_nodes_template)
    elif type =='path_tnm':
        queryTime = crc_tool_spec.get_path_tnm_fn(path_tnm_template)
    elif type == 'tumor_size':
        queryTime = crc_tool_spec.get_tumor_size_fn(tumor_size_template)
    else:
        raise Exception('Not support yet!')

    results = {}
        
    results["queryTime"] = queryTime

    with open(analysis_file_path+'.json', 'w', encoding="utf-8") as the_file:
        the_file.write(json.dumps(results, ensure_ascii=False, indent=2))
    pickle.dump(results, open(analysis_file_path, "wb"))

In [ ]:
def run_experiment(registry_type, one_match_mode = True):
    target_names = {}
    if 'NA' not in target_names:
        target_names['NA'] = len(target_names)

    for i, k in enumerate(tqdm.tqdm(patient_list.keys())):
        golds = list(patient_list[k][2].values())[0]
        if HospitalID == 'CMUH' and registry_type == 'histology':
            tmp_golds = [list(patient_list[k][2].values())[0][0][:4]].copy()
        elif HospitalID == 'CMUH' and registry_type == 'tumor_size':
            tmp_golds = [list(patient_list[k][2].values())[0][0][:-1] + "." + list(patient_list[k][2].values())[0][0][-1]]
        else:
            tmp_golds = list(patient_list[k][2].values())[0].copy()
        for ans in tmp_golds:
            if ans not in target_names:
                target_names[ans] = len(target_names)

        for idx, doc in enumerate(patient_list[k][1]):
            nodes = node_parser.get_nodes_from_documents([doc])
            build_analyze_per_doc(doc, idx, registry_type, nodes, golds)

In [ ]:
def run_experiments(registry_types_name, target_outcome_list, one_match_mode = True):
    target_names = {}

    for i, k in enumerate(tqdm.tqdm(patient_list.keys())):
        golds = patient_list[k][2]
        tmp_golds = {}
        for gold_k in golds.keys():
            tmp_golds[gold_k.lower()] = golds[gold_k].copy()
        # update names
        for type in tmp_golds.keys():
            if type not in target_names:
                target_names[type]= {}
                target_names[type]['NA'] = len(target_names[type])            
    
            for ans in tmp_golds[type]:
                if ans not in target_names[type]:
                    target_names[type][ans] = len(target_names[type])
        
        for idx, doc in enumerate(patient_list[k][1]):
            nodes = node_parser.get_nodes_from_documents([doc])
            build_analyze_per_doc(doc, idx, registry_types_name, nodes, golds)

In [ ]:
golds, patient_list = select_registry_item(gold_ans, 'HISTOLOGY', ignore_list)
golds.drop_duplicates()
run_experiment('histology')

In [ ]:
golds, patient_list =  select_registry_item(gold_ans, 'GRADE', ignore_list)
golds.drop_duplicates()
run_experiment('grade')

In [ ]:
if HospitalID not in {"KAFGH_C", "KAFGH_L"}:
    golds, patient_list =  select_registry_item(gold_ans, 'TUMOR_SIZE', ignore_list)
    golds.drop_duplicates()
    run_experiment('tumor_size')

In [ ]:
golds, patient_list =  select_registry_item(gold_ans, 'NODE_EXAMINED', ignore_list)
golds.drop_duplicates()
run_experiment('examined_nodes')

In [ ]:
golds, patient_list =  select_registry_item(gold_ans, 'NODE_POSITIVE', ignore_list)
golds.drop_duplicates()
run_experiment('positive_nodes')

In [ ]:
golds, patient_list =  select_registry_items(gold_ans, ['PATH_T', 'PATH_N', 'PATH_M'], ignore_list)
golds.drop_duplicates()
run_experiments('path_tnm', ['path_t', 'path_n', 'path_m'])